# **Component I. Vector Indexing and Embedding**

---

### **Cell 1: Install Required Libraries**

**Purpose:**  

Prepares the Google Colab environment by installing all the tools needed for PDF processing, text chunking, embeddings, FAISS searching, Meta-Llama-3-8B-Instruct processing, Hugging Face authentication, 4-bit model loading, and Excel file creation.

**Input:**  

- Google Colab Python environment

- Internet connection to download the required packages

- No PDF file or question is required yet

**Output:**  

- All required Python packages are installed

- FAISS, Sentence Transformers, PyPDF, Transformers, Pandas, OpenPyXL, Accelerate, BitsAndBytes, and Hugging Face Hub become available

- The environment is ready for the next cells

---

**Line-by-line breakdown:**

- `!pip install -q \` — Tells Google Colab to run a package installation command. The `!` means the line is executed as a system command instead of normal Python code. `pip install` downloads and installs packages, while `-q` reduces the amount of installation messages shown.

- `faiss-cpu \` — Installs FAISS, which is used later to search for document chunks that are most similar to a question. The CPU version allows FAISS to run using the computer's processor.

- `sentence-transformers \` — Installs the tool used to change text into numerical representations called embeddings. These numbers allow the computer to compare the meaning of different texts.

- `pypdf \` — Installs the library used to open PDF files and extract text from their pages.

- `transformers \` — Installs the Hugging Face Transformers library. It is used in Component II to load the Meta-Llama-3-8B-Instruct model and its tokenizer.

- `sentencepiece \` — Installs a supporting tool required by some language models for dividing text into smaller pieces that the model can understand.

- `pandas \` — Installs Pandas, which is used to organize chunk information into a table.

- `openpyxl \` — Installs the library used to create and format the Excel file containing the generated chunks.

- `accelerate \` — Installs the Hugging Face Accelerate library. It helps manage how the large Llama 3 model is placed and loaded on the available GPU.

- `bitsandbytes \` — Installs the library used to load Meta-Llama-3-8B-Instruct using 4-bit quantization. This reduces the amount of GPU memory required by the model.

- `huggingface_hub` — Installs the Hugging Face Hub library. It allows the notebook to authenticate using a Hugging Face access token so that the official Meta-Llama-3-8B-Instruct model can be accessed.

---

### **Why this step is important:**

The rest of the notebook depends on these packages. The additional `accelerate`, `bitsandbytes`, and `huggingface_hub` packages are needed because Meta-Llama-3-8B-Instruct is much larger than the previous FLAN-T5 model and requires Hugging Face authentication and more efficient GPU memory usage.

In Google Colab, packages may need to be installed again after the runtime is restarted.

---

In [ ]:
# ============================================================
# RAG DOCUMENT PROCESSING PIPELINE
# PDF EXTRACTION, CHUNKING, EMBEDDINGS, FAISS, AND RETRIEVAL
# WITH DOWNLOADABLE EXCEL CHUNK INSPECTION
# Google Colab implementation
# ============================================================


# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    transformers \
    sentencepiece \
    pandas \
    openpyxl \
    accelerate \
    bitsandbytes \
    huggingface_hub

---
### **Cell 2: Import Libraries**

**Purpose:**  
Loads the installed libraries and specific tools into the current Python program so they can be used throughout Component I.

**Input:**  
- Packages installed in Cell 1

**Output:**  
- PDF reading tools become available
- File upload and download tools become available
- Embedding and FAISS tools become available
- Data and Excel processing tools become available

---

**Line-by-line breakdown:**

- `from google.colab import files` — Loads Google Colab's file tool. It is used to upload the PDF and download the generated Excel file.

- `from pypdf import PdfReader` — Loads `PdfReader`, which is used to open the PDF and read its pages.

- `from sentence_transformers import SentenceTransformer` — Loads the tool that changes text into numerical embeddings.

- `import faiss` — Loads FAISS for storing and searching the document embeddings.

- `import numpy as np` — Loads NumPy and gives it the shorter name `np`. It is used for numerical operations such as calculating the average chunk length.

- `import pandas as pd` — Loads Pandas as `pd`. It is used to organize the chunk information into rows and columns.

- `import pickle` — Loads Python's Pickle tool, which is used to save Python data such as chunks and configuration settings into files.

- `import re` — Loads Python's text pattern tool. It is used to clean unnecessary spaces and line breaks from extracted PDF text.

- `import os` — Loads tools for working with filenames and file paths.

- `from openpyxl.styles import Font, PatternFill, Alignment` — Loads tools used to format the generated Excel workbook.

- `Font` — Controls the appearance of text in Excel cells.

- `PatternFill` — Controls the background color of Excel cells.

- `Alignment` — Controls how text is positioned and wrapped inside Excel cells.

---

### **Why this step is important:**

Installing a library makes it available in Google Colab, but importing it allows the current Python program to actually use its functions and features.

---

In [ ]:
# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np
import pandas as pd
import pickle
import re
import os

from openpyxl.styles import Font, PatternFill, Alignment

---
### **Cell 3: Set Indexing Parameters**

**Purpose:**  
Defines the main settings used when dividing the PDF into chunks and searching the document.

**Input:**  
- Values manually selected by the programmer

**Output:**  
- Chunk size is set to 1000 characters
- Chunk overlap is set to 200 characters
- Top-K is set to 3 results
- Embedding model is set to `all-MiniLM-L6-v2`

---

**Line-by-line breakdown:**

- `CHUNK_SIZE = 1000` — Sets each chunk to contain up to approximately 1000 characters.

- `CHUNK_OVERLAP = 200` — Sets 200 characters to repeat between neighboring chunks. This helps prevent information near a chunk boundary from being separated completely.

- `TOP_K = 3` — Tells the retrieval system to return the three best matching chunks.

- `EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"` — Stores the name of the model that will convert the chunks and questions into numerical embeddings.

---

### **Why this step is important:**

These settings directly affect how the document is divided and searched. In the experiment, changing the chunk size and overlap allows different configurations to be compared.

---

In [138]:
# ============================================================
# 3. INDEXING PARAMETERS
# ============================================================

# Change these values to experiment with chunking.

CHUNK_SIZE = 600
CHUNK_OVERLAP = 100
TOP_K = 3

# Embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

---
### **Cell 4: Validate Chunk Parameters**

**Purpose:**  
Checks whether the chunking and retrieval settings contain valid values before the notebook continues.

**Input:**  
- `CHUNK_SIZE`
- `CHUNK_OVERLAP`
- `TOP_K`
- `EMBEDDING_MODEL_NAME`

**Output:**  
- Stops the program if an invalid setting is found
- Displays the current indexing configuration if all values are valid

---

**Line-by-line breakdown:**

- `if CHUNK_SIZE <= 0:` — Checks whether the chunk size is zero or negative.

- `raise ValueError("CHUNK_SIZE must be greater than 0.")` — Stops the program and displays an error if the chunk size is invalid.

- `if CHUNK_OVERLAP < 0:` — Checks whether the overlap is negative.

- `raise ValueError("CHUNK_OVERLAP cannot be negative.")` — Stops the program if a negative overlap is entered.

- `if CHUNK_OVERLAP >= CHUNK_SIZE:` — Checks whether the overlap is equal to or larger than the chunk itself.

- `raise ValueError(...)` — Stops the program when the overlap is too large.

- `"CHUNK_OVERLAP must be smaller than CHUNK_SIZE."` — Explains the required relationship between the two settings.

- `if TOP_K <= 0:` — Checks whether the requested number of search results is zero or negative.

- `raise ValueError("TOP_K must be greater than 0.")` — Stops the program if `TOP_K` is invalid.

- `print("=" * 70)` — Prints 70 equal signs to create a visual separator.

- `print("RAG INDEXING CONFIGURATION")` — Displays the title of the current section.

- `print("Chunk size       :", CHUNK_SIZE)` — Displays the selected chunk size.

- `print("Chunk overlap    :", CHUNK_OVERLAP)` — Displays the selected overlap.

- `print("Top-K             :", TOP_K)` — Displays the number of retrieval results requested.

- `print("Embedding model   :", EMBEDDING_MODEL_NAME)` — Displays the selected embedding model.

---

### **Why this step is important:**

Checking the values before processing the document prevents incorrect settings from causing problems later in the program.

---

In [140]:
# ============================================================
# 4. VALIDATE CHUNK PARAMETERS
# ============================================================

if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be greater than 0.")

if CHUNK_OVERLAP < 0:
    raise ValueError("CHUNK_OVERLAP cannot be negative.")

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError(
        "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
    )

if TOP_K <= 0:
    raise ValueError("TOP_K must be greater than 0.")


print("=" * 70)
print("RAG INDEXING CONFIGURATION")
print("=" * 70)

print("Chunk size       :", CHUNK_SIZE)
print("Chunk overlap    :", CHUNK_OVERLAP)
print("Top-K             :", TOP_K)
print("Embedding model   :", EMBEDDING_MODEL_NAME)

RAG INDEXING CONFIGURATION
Chunk size       : 600
Chunk overlap    : 100
Top-K             : 3
Embedding model   : all-MiniLM-L6-v2


---
### **Cell 5: Upload PDF Document**

**Purpose:**  
Allows the user to upload the PDF document that will be processed and checks whether the uploaded file is actually a PDF.

**Input:**  
- PDF file selected by the user

**Output:**  
- Uploaded file information stored in `uploaded`
- Selected PDF filename stored in `pdf_file`
- Error message if no file or a non-PDF file is provided

---

**Line-by-line breakdown:**

- `print("\n")` — Adds an empty line before displaying the section.

- `print("=" * 70)` — Displays a separator.

- `print("UPLOAD DOCUMENT")` — Displays the name of the current process.

- `uploaded = files.upload()` — Opens Google Colab's file upload window and stores the uploaded file information.

- `if len(uploaded) == 0:` — Checks whether the user uploaded any file.

- `raise RuntimeError("No file was uploaded.")` — Stops the program if no file was selected.

- `# Use the first uploaded PDF` — Explains that only the first uploaded file will be used.

- `pdf_file = next(iter(uploaded.keys()))` — Gets the filename of the first uploaded file and stores it in `pdf_file`.

- `uploaded.keys()` — Gets the filenames contained in the uploaded files.

- `iter(...)` — Allows Python to go through those filenames.

- `next(...)` — Gets the first filename.

- `if not pdf_file.lower().endswith(".pdf"):` — Checks whether the filename ends with `.pdf`. `.lower()` allows both `.pdf` and `.PDF` to be accepted.

- `raise ValueError(...)` — Stops the program if the uploaded file is not a PDF.

- `print("\nSelected file:")` — Displays a label for the selected file.

- `print(pdf_file)` — Displays the name of the PDF that will be processed.

---

### **Why this step is important:**

The following cells require a valid PDF document. Checking the file before processing prevents the PDF reader from receiving an unsupported file.

---


In [ ]:
# ============================================================
# 5. UPLOAD PDF DOCUMENT
# ============================================================

print("\n")
print("=" * 70)
print("UPLOAD DOCUMENT")
print("=" * 70)

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No file was uploaded.")

# Use the first uploaded PDF
pdf_file = next(iter(uploaded.keys()))

if not pdf_file.lower().endswith(".pdf"):
    raise ValueError(
        "The uploaded file is not a PDF. Please upload a PDF."
    )

print("\nSelected file:")
print(pdf_file)



UPLOAD DOCUMENT


Saving AO-2026-0005-Guidelines-on-Differentiated-Service-Delivery-for-Pre-Exposure-Prophylaxis-PrEP-and-Post-Exposure-Prophylaxis-PEP-in-the-Prevention-of-HIV-in-the-Philippines-1.pdf to AO-2026-0005-Guidelines-on-Differentiated-Service-Delivery-for-Pre-Exposure-Prophylaxis-PrEP-and-Post-Exposure-Prophylaxis-PEP-in-the-Prevention-of-HIV-in-the-Philippines-1 (1).pdf

Selected file:
AO-2026-0005-Guidelines-on-Differentiated-Service-Delivery-for-Pre-Exposure-Prophylaxis-PrEP-and-Post-Exposure-Prophylaxis-PEP-in-the-Prevention-of-HIV-in-the-Philippines-1 (1).pdf


---
### **Cell 6: Extract Text From PDF**

**Purpose:**  
Opens the uploaded PDF, extracts text from each page, cleans unnecessary spacing, and keeps the original page number.

**Input:**  
- `pdf_file`

**Output:**  
- Opened PDF stored in `reader`
- Extracted page information stored in `pages`
- Each saved page contains its page number and cleaned text

---

**Line-by-line breakdown:**

- `reader = PdfReader(pdf_file)` — Opens the selected PDF using `PdfReader`.

- `pages = []` — Creates an empty list where extracted page information will be stored.

- `for page_number, page in enumerate(reader.pages, start=1):` — Goes through every PDF page. `start=1` makes page numbering begin at 1.

- `page_text = page.extract_text()` — Extracts readable text from the current page.

- `if page_text:` — Continues only if some text was successfully extracted.

- `# Normalize whitespace` — Explains that the following operation cleans unnecessary spacing.

- `page_text = re.sub(...)` — Uses a text replacement operation to clean repeated spaces, tabs, and line breaks.

- `r"\s+"` — Represents one or more spacing characters.

- `" "` — Replaces those repeated spacing characters with one normal space.

- `.strip()` — Removes unnecessary spaces from the beginning and end of the text.

- `if page_text:` — Checks again that the cleaned text is not empty.

- `pages.append({` — Adds the current page information to the `pages` list.

- `"page": page_number` — Saves the original PDF page number.

- `"text": page_text` — Saves the cleaned text from that page.

- `print("PDF pages:", len(reader.pages))` — Displays the total number of pages in the PDF.

- `print("Pages containing text:", len(pages))` — Displays how many pages contained extractable text.

---

### **Why this step is important:**

The search system needs actual text rather than the visual PDF page. Saving the page number also allows retrieved information to be connected back to its original location.

---


In [ ]:
# ============================================================
# 6. EXTRACT TEXT FROM PDF
# ============================================================

print("\n")
print("=" * 70)
print("EXTRACTING PDF TEXT")
print("=" * 70)

reader = PdfReader(pdf_file)

pages = []

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if page_text:

        # Normalize whitespace
        page_text = re.sub(
            r"\s+",
            " ",
            page_text
        ).strip()

        if page_text:
            pages.append({
                "page": page_number,
                "text": page_text
            })


print("PDF pages:", len(reader.pages))
print("Pages containing text:", len(pages))



EXTRACTING PDF TEXT
PDF pages: 32
Pages containing text: 32


---
### **Cell 7: Check Whether Text Was Extracted**

**Purpose:**  
Checks whether the PDF produced any readable text before continuing.

**Input:**  
- `pages`

**Output:**  
- Program continues if text exists
- Program stops if no readable text was extracted

---

**Line-by-line breakdown:**

- `if len(pages) == 0:` — Checks whether the list of extracted pages is empty.

- `raise RuntimeError(...)` — Stops the program when no readable pages exist.

- `"No text could be extracted from the PDF. "` — Explains the main problem.

- `"The PDF may consist of scanned images and require OCR."` — Explains that a scanned PDF may contain images of text instead of actual selectable text.

---

### **Why this step is important:**

Chunking, embeddings, and retrieval cannot work without text. This check prevents the program from continuing with an empty document.

---

In [ ]:
# ============================================================
# 7. CHECK WHETHER TEXT WAS EXTRACTED
# ============================================================

if len(pages) == 0:
    raise RuntimeError(
        "No text could be extracted from the PDF. "
        "The PDF may consist of scanned images and require OCR."
    )

---
### **Cell 8: Create Document Chunks**

**Purpose:**  
Divides the extracted PDF text into smaller pieces called chunks while maintaining overlap between neighboring chunks.

**Input:**  
- `pages`
- `CHUNK_SIZE`
- `CHUNK_OVERLAP`

**Output:**  
- `chunks`
- Each chunk contains a chunk ID, page number, and text

---

**Line-by-line breakdown:**

- `def create_chunks(pages, chunk_size, chunk_overlap):` — Creates a reusable function for dividing pages into chunks.

- `""" ... """` — Provides a description of what the function does.

- `chunks = []` — Creates an empty list where generated chunks will be stored.

- `chunk_id = 0` — Starts the chunk numbering at zero.

- `step_size = chunk_size - chunk_overlap` — Calculates how far the program moves forward after creating each chunk.

- `for page_data in pages:` — Goes through each extracted PDF page.

- `page_number = page_data["page"]` — Gets the original page number.

- `page_text = page_data["text"]` — Gets the cleaned text from that page.

- `start = 0` — Starts at the first character of the page.

- `while start < len(page_text):` — Continues creating chunks while text remains.

- `end = start + chunk_size` — Calculates the ending position of the current chunk.

- `chunk_text = page_text[start:end].strip()` — Gets the text between the starting and ending positions and removes extra spaces.

- `if chunk_text:` — Makes sure the chunk contains actual text.

- `chunks.append({ ... })` — Adds the new chunk and its information to the list.

- `"chunk_id": chunk_id` — Saves the unique chunk number.

- `"page": page_number` — Saves the page where the chunk came from.

- `"text": chunk_text` — Saves the actual text.

- `chunk_id += 1` — Adds one to the chunk ID for the next chunk.

- `start += step_size` — Moves the starting position forward while maintaining the selected overlap.

- `return chunks` — Returns the completed list of chunks.

- `chunks = create_chunks(...)` — Runs the function using the extracted pages and selected chunk settings.

---

### **Why this step is important:**

Smaller chunks allow the search system to find specific sections of the document instead of comparing the question with the entire PDF at once. The chunk size and overlap are also the main settings being tested in the experiment.

---

In [142]:
# ============================================================
# 8. CREATE DOCUMENT CHUNKS
# ============================================================

def create_chunks(
    pages,
    chunk_size,
    chunk_overlap
):
    """
    Split each page into character-based chunks.

    Chunking is performed independently for each page.
    Overlap is maintained between consecutive chunks
    from the same page.
    """

    chunks = []
    chunk_id = 0

    step_size = chunk_size - chunk_overlap

    for page_data in pages:

        page_number = page_data["page"]
        page_text = page_data["text"]

        start = 0

        while start < len(page_text):

            end = start + chunk_size

            chunk_text = page_text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "chunk_id": chunk_id,
                    "page": page_number,
                    "text": chunk_text
                })

                chunk_id += 1

            # Move forward while maintaining overlap
            start += step_size

    return chunks


chunks = create_chunks(
    pages,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

---
### **Cell 9: Display Chunking Information**

**Purpose:**  
Checks the generated chunks and displays basic information about their sizes.

**Input:**  
- `chunks`

**Output:**  
- Number of generated chunks
- Average chunk length
- Shortest chunk
- Longest chunk

---

**Line-by-line breakdown:**

- `print("Number of chunks:", len(chunks))` — Counts and displays the total number of generated chunks.

- `if len(chunks) == 0:` — Checks whether chunk creation produced zero chunks.

- `raise RuntimeError(...)` — Stops the program if no chunks were created.

- `chunk_lengths = [` — Begins creating a list containing the size of every chunk.

- `len(chunk["text"])` — Counts the characters in the current chunk.

- `for chunk in chunks` — Repeats the character count for every chunk.

- `np.mean(chunk_lengths)` — Calculates the average number of characters in the chunks.

- `round(..., 2)` — Rounds the average to two decimal places.

- `min(chunk_lengths)` — Finds the smallest chunk.

- `max(chunk_lengths)` — Finds the largest chunk.

- The `print(...)` statements display the calculated information.

---

### **Why this step is important:**

This gives a quick summary of how the selected chunk settings affected the PDF. It is useful when comparing different chunk-size and overlap configurations.

---

In [143]:
# ============================================================
# 9. DISPLAY CHUNKING INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("CHUNKING RESULTS")
print("=" * 70)

print("Number of chunks:", len(chunks))

if len(chunks) == 0:
    raise RuntimeError(
        "No chunks were generated. Check the extracted PDF text."
    )

chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print(
    "Average characters per chunk:",
    round(np.mean(chunk_lengths), 2)
)

print(
    "Shortest chunk:",
    min(chunk_lengths),
    "characters"
)

print(
    "Longest chunk:",
    max(chunk_lengths),
    "characters"
)



CHUNKING RESULTS
Number of chunks: 133
Average characters per chunk: 508.57
Shortest chunk: 13 characters
Longest chunk: 600 characters


---
### **Cell 10: Export Generated Chunks to Excel**

**Purpose:**  
Creates a formatted Excel workbook containing every generated chunk so the chunks can be manually inspected.

**Input:**  
- `chunks`
- `pdf_file`
- `CHUNK_SIZE`
- `CHUNK_OVERLAP`

**Output:**  
- Excel file containing all generated chunks
- Chunk IDs, page numbers, character counts, settings, and chunk text
- Automatically downloaded Excel workbook

---

**Line-by-line breakdown:**

- `chunk_records = []` — Creates an empty list for the information that will become Excel rows.

- `for chunk in chunks:` — Goes through every generated chunk.

- `chunk_records.append({ ... })` — Creates one record for the current chunk.

- `"Chunk_ID": chunk["chunk_id"]` — Saves the chunk ID.

- `"Source_Document": pdf_file` — Saves the original PDF filename.

- `"Page_Number": chunk["page"]` — Saves the original page number.

- `"Character_Count": len(chunk["text"])` — Saves the number of characters in the chunk.

- `"Chunk_Size_Setting": CHUNK_SIZE` — Records the chunk-size setting used.

- `"Chunk_Overlap_Setting": CHUNK_OVERLAP` — Records the overlap setting used.

- `"Chunk_Text": chunk["text"]` — Saves the actual chunk text.

- `chunks_df = pd.DataFrame(chunk_records)` — Converts the records into a table.

- `os.path.basename(pdf_file)` — Gets the PDF filename without its folder path.

- `os.path.splitext(...)[0]` — Removes the `.pdf` extension.

- `EXCEL_FILE = f"{base_name}_chunks_inspection.xlsx"` — Creates the Excel filename.

- `with pd.ExcelWriter(EXCEL_FILE, engine="openpyxl") as writer:` — Opens an Excel-writing process using OpenPyXL.

- `chunks_df.to_excel(...)` — Writes the chunk table into the Excel workbook.

- `sheet_name="Generated_Chunks"` — Names the worksheet `Generated_Chunks`.

- `index=False` — Prevents Pandas from adding an unnecessary number column.

- `worksheet = writer.sheets["Generated_Chunks"]` — Gets the worksheet so it can be formatted.

- `for cell in worksheet[1]:` — Goes through every cell in the header row.

- `cell.font = Font(...)` — Makes the header text bold and white.

- `cell.fill = PatternFill(...)` — Adds a solid background color to the header.

- `cell.alignment = Alignment(...)` — Centers and wraps the header text.

- `column_widths = {...}` — Stores the desired width of each Excel column.

- `for column, width in column_widths.items():` — Goes through each column and its width.

- `worksheet.column_dimensions[column].width = width` — Applies the selected width.

- `worksheet.iter_rows(...)` — Goes through the data rows.

- `vertical="top"` — Places cell text at the top.

- `wrap_text=(cell.column == 7)` — Wraps the long chunk text in column G.

- `worksheet.freeze_panes = "A2"` — Keeps the header visible while scrolling.

- `worksheet.auto_filter.ref = worksheet.dimensions` — Adds filtering controls to the worksheet.

- `worksheet.row_dimensions[1].height = 32` — Changes the height of the header row.

- `print("Excel file created:", EXCEL_FILE)` — Displays the generated filename.

- `print("Chunks exported:", len(chunks_df))` — Displays the number of exported chunks.

- `.head(10)` — Selects the first 10 rows for a preview.

- `.to_string(index=False)` — Displays the preview without extra row numbers.

- `files.download(EXCEL_FILE)` — Downloads the Excel workbook through Google Colab.

---

### **Why this step is important:**

The Excel workbook makes it possible to manually inspect how the document was divided. This is especially useful when evaluating different chunk-size and overlap settings.

---

In [144]:
# ============================================================
# 10. EXPORT GENERATED CHUNKS TO EXCEL
# ============================================================

print("\n")
print("=" * 70)
print("EXPORTING GENERATED CHUNKS TO EXCEL")
print("=" * 70)

# Prepare one Excel row for every generated chunk.
chunk_records = []

for chunk in chunks:

    chunk_records.append({
        "Chunk_ID": chunk["chunk_id"],
        "Source_Document": pdf_file,
        "Page_Number": chunk["page"],
        "Character_Count": len(chunk["text"]),
        "Chunk_Size_Setting": CHUNK_SIZE,
        "Chunk_Overlap_Setting": CHUNK_OVERLAP,
        "Chunk_Text": chunk["text"]
    })

chunks_df = pd.DataFrame(chunk_records)

# Create an output filename based on the source PDF.
base_name = os.path.splitext(
    os.path.basename(pdf_file)
)[0]

EXCEL_FILE = f"{base_name}_chunks_inspection.xlsx"

# Create and format the Excel workbook.
with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    chunks_df.to_excel(
        writer,
        sheet_name="Generated_Chunks",
        index=False
    )

    worksheet = writer.sheets["Generated_Chunks"]

    # Format the header row.
    for cell in worksheet[1]:

        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # Set column widths.
    column_widths = {
        "A": 12,
        "B": 35,
        "C": 14,
        "D": 18,
        "E": 20,
        "F": 23,
        "G": 80
    }

    for column, width in column_widths.items():
        worksheet.column_dimensions[column].width = width

    # Format rows for easier inspection.
    for row in worksheet.iter_rows(
        min_row=2,
        max_row=worksheet.max_row
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=(cell.column == 7)
            )

    # Keep headers visible while scrolling.
    worksheet.freeze_panes = "A2"

    # Enable filtering.
    worksheet.auto_filter.ref = worksheet.dimensions

    # Set header height.
    worksheet.row_dimensions[1].height = 32

print("Excel file created:", EXCEL_FILE)
print("Chunks exported:", len(chunks_df))
print("Excel worksheet: Generated_Chunks")

print("\nExcel export preview:")

print(
    chunks_df[
        [
            "Chunk_ID",
            "Page_Number",
            "Character_Count"
        ]
    ].head(10).to_string(index=False)
)

# Automatically download the Excel workbook in Google Colab.
files.download(EXCEL_FILE)



EXPORTING GENERATED CHUNKS TO EXCEL
Excel file created: AO-2026-0005-Guidelines-on-Differentiated-Service-Delivery-for-Pre-Exposure-Prophylaxis-PrEP-and-Post-Exposure-Prophylaxis-PEP-in-the-Prevention-of-HIV-in-the-Philippines-1 (1)_chunks_inspection.xlsx
Chunks exported: 133
Excel worksheet: Generated_Chunks

Excel export preview:
 Chunk_ID  Page_Number  Character_Count
        0            1              600
        1            1              599
        2            1              599
        3            1              599
        4            1              600
        5            1              162
        6            2              600
        7            2              600
        8            2              600
        9            2              600


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### **Cell 11: Display Sample Chunks**

**Purpose:**  
Displays the first three chunks directly in Google Colab for a quick inspection.

**Input:**  
- `chunks`

**Output:**  
- First three chunk IDs
- Page numbers
- Character counts
- Actual chunk text

---

**Line-by-line breakdown:**

- `for chunk in chunks[:3]:` — Goes through only the first three chunks.

- `print(f"\nChunk ID : {chunk['chunk_id']}")` — Displays the chunk ID.

- `print(f"Page     : {chunk['page']}")` — Displays the original PDF page.

- `print(f"Characters: {len(chunk['text'])}")` — Displays the number of characters.

- `print("-" * 70)` — Displays a separator.

- `print(chunk["text"])` — Displays the actual text contained in the chunk.

---

### **Why this step is important:**

This provides a quick way to check whether the chunks contain readable and useful text without opening the Excel file.

---

In [145]:
# ============================================================
# 11. DISPLAY SAMPLE CHUNKS
# ============================================================

print("\n")
print("=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)

for chunk in chunks[:3]:

    print(f"\nChunk ID : {chunk['chunk_id']}")
    print(f"Page     : {chunk['page']}")
    print(f"Characters: {len(chunk['text'])}")

    print("-" * 70)

    print(chunk["text"])



SAMPLE CHUNKS

Chunk ID : 0
Page     : 1
Characters: 600
----------------------------------------------------------------------
Republic ofthe Philippines DEPARTMENT OF HEALTH Office of the Secretary BAGONG PILIPINAS MAR 23 2025 ADMINISTRATIVE ORDER No. 2026 -_0005 SUBJECT: Guidelines on Differentiated Service Deliver: DSD) _ for Pre-Exposure Prophylaxis (PrEP) and Post-Exposure Prophylaxis (PEP) in the Prevention ofHIV in the Philippines I. RATIONALE The Philippines remains to have a less than one percent HIV prevalence. Despite the low prevalence, there is a concentrated epidemic within key populations. The country has been reported to have the fastest-growing epidemic in Asia and the Pacific Region with a 550% incre

Chunk ID : 1
Page     : 1
Characters: 599
----------------------------------------------------------------------
been reported to have the fastest-growing epidemic in Asia and the Pacific Region with a 550% increase in annual new infections between 2010 and 2024. As o

---
### **Cell 12: Load Embedding Model**

**Purpose:**  
Loads the model that will convert document text into numerical representations that can be compared by FAISS.

**Input:**  
- `EMBEDDING_MODEL_NAME`

**Output:**  
- Loaded embedding model stored in `embedding_model`

---

**Line-by-line breakdown:**

- `embedding_model = SentenceTransformer(` — Begins loading a Sentence Transformer model.

- `EMBEDDING_MODEL_NAME` — Provides the model name defined earlier.

- `)` — Completes the model-loading command.

- `print("Embedding model:", EMBEDDING_MODEL_NAME)` — Displays the name of the loaded model.

---

### **Why this step is important:**

FAISS searches numbers instead of directly understanding sentences. The embedding model converts text into numbers that represent its meaning.

---

In [ ]:
# ============================================================
# 12. LOAD EMBEDDING MODEL
# ============================================================

print("\n")
print("=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)



LOADING EMBEDDING MODEL


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model: all-MiniLM-L6-v2


---
### **Cell 13: Extract Chunk Text**

**Purpose:**  
Creates a separate list containing only the text from each generated chunk.

**Input:**  
- `chunks`

**Output:**  
- List of chunk texts stored in `chunk_texts`

---

**Line-by-line breakdown:**

- `chunk_texts = [` — Starts creating a new list.

- `chunk["text"]` — Gets the text from the current chunk.

- `for chunk in chunks` — Repeats the operation for every generated chunk.

- `]` — Completes the list.

---

### **Why this step is important:**

The embedding model only needs the actual text. Information such as page numbers and chunk IDs does not need to be converted into embeddings.

---

In [146]:
# ============================================================
# 13. EXTRACT CHUNK TEXT
# ============================================================

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

---
### **Cell 14: Create Document Embeddings**

**Purpose:**  
Converts every document chunk into numerical values that FAISS can store and compare.

**Input:**  
- `chunk_texts`
- `embedding_model`

**Output:**  
- Numerical embeddings stored in `embeddings`
- One embedding for every document chunk

---

**Line-by-line breakdown:**

- `embeddings = embedding_model.encode(` — Sends the document chunks to the embedding model.

- `chunk_texts` — Provides all chunk texts.

- `convert_to_numpy=True` — Returns the results as a NumPy array.

- `normalize_embeddings=True` — Adjusts the embeddings to a common scale so similarity comparison works properly.

- `show_progress_bar=True` — Displays progress while the chunks are processed.

- `embeddings = embeddings.astype("float32")` — Changes the numbers into the `float32` format required by FAISS.

- `print(embeddings.shape)` — Displays the size of the completed embedding table.

For example, `(100, 384)` means 100 chunks were created and each chunk is represented using 384 numbers.

---

### **Why this step is important:**

This converts the document from readable text into the numerical form required for similarity searching.

---

In [147]:
# ============================================================
# 14. CREATE DOCUMENT EMBEDDINGS
# ============================================================

print("\n")
print("=" * 70)
print("CREATING DOCUMENT EMBEDDINGS")
print("=" * 70)

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# FAISS expects float32 vectors.
embeddings = embeddings.astype("float32")

print("\nEmbedding matrix shape:")
print(embeddings.shape)



CREATING DOCUMENT EMBEDDINGS


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding matrix shape:
(133, 384)


---
### **Cell 15: Create FAISS Vector Store**

**Purpose:**  
Creates the FAISS search index and stores all document embeddings inside it.

**Input:**  
- `embeddings`

**Output:**  
- FAISS index stored in `index`
- All document embeddings added to the index

---

**Line-by-line breakdown:**

- `embedding_dimension = embeddings.shape[1]` — Gets the number of values used to represent each embedding.

- `# Inner Product on normalized vectors is equivalent to cosine similarity.` — Explains that because the embeddings were normalized earlier, the selected FAISS comparison method can be used to measure similarity.

- `index = faiss.IndexFlatIP(` — Creates a FAISS index using inner-product comparison.

- `embedding_dimension` — Tells FAISS how many numerical values each embedding contains.

- `index.add(embeddings)` — Adds all document embeddings to the FAISS index.

- `print("Embedding dimension:", embedding_dimension)` — Displays the number of values in each embedding.

- `print("Vectors stored:", index.ntotal)` — Displays how many embeddings are currently stored.

---

### **Why this step is important:**

This creates the actual searchable vector store. Without the FAISS index, the system would not be able to quickly find document chunks that are similar to a question.

---

In [148]:
# ============================================================
# 15. CREATE FAISS VECTOR STORE
# ============================================================

print("\n")
print("=" * 70)
print("CREATING FAISS VECTOR STORE")
print("=" * 70)

embedding_dimension = embeddings.shape[1]

# Inner Product on normalized vectors is equivalent
# to cosine similarity.
index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors stored:", index.ntotal)



CREATING FAISS VECTOR STORE
Embedding dimension: 384
Vectors stored: 133


---
### **Cell 16: Create Chunk Lookup**

**Purpose:**  
Creates a connection between each FAISS embedding position and its original document chunk.

**Input:**  
- `chunks`

**Output:**  
- Lookup table stored in `chunk_lookup`

---

**Line-by-line breakdown:**

- `# The embedding row index corresponds to the position of the chunk in the chunks list.` — Explains that each embedding and chunk share the same position.

- `chunk_lookup = {` — Starts creating a lookup table.

- `position: chunk` — Uses the chunk's position as the key and the complete chunk information as the value.

- `for position, chunk in enumerate(chunks)` — Goes through every chunk while also keeping track of its position.

- `}` — Completes the lookup table.

---

### **Why this step is important:**

FAISS returns the positions of matching embeddings. This lookup table allows those positions to be converted back into useful information such as the chunk text, page number, and chunk ID.

---

In [149]:
# ============================================================
# 16. CREATE CHUNK LOOKUP
# ============================================================

# The embedding row index corresponds to the position
# of the chunk in the chunks list.

chunk_lookup = {
    position: chunk
    for position, chunk in enumerate(chunks)
}

---
### **Cell 17: Define Retrieval Function**

**Purpose:**  
Creates the function that accepts a question, searches the FAISS index, and returns the most similar document chunks.

**Input:**  
- User question stored in `query`
- Number of requested results stored in `top_k`

**Output:**  
- Ranked list of matching chunks
- Chunk ID
- Page number
- Similarity score
- Chunk text

---

**Line-by-line breakdown:**

- `def retrieve_documents(query, top_k=3):` — Creates a reusable retrieval function. By default, it returns three results.

- `""" ... """` — Describes what the function does and what information it returns.

- `if not query or not query.strip():` — Checks whether the question is empty or contains only spaces.

- `raise ValueError("The query cannot be empty.")` — Stops the function when an empty question is provided.

- `if top_k <= 0:` — Checks whether the requested number of results is valid.

- `raise ValueError("top_k must be greater than 0.")` — Stops the function if `top_k` is zero or negative.

- `query_embedding = embedding_model.encode(...)` — Converts the user's question into an embedding using the same model used for the document.

- `[query]` — Places the question inside a list because the model expects a collection of texts.

- `convert_to_numpy=True` — Returns the question embedding as a NumPy array.

- `normalize_embeddings=True` — Normalizes the question embedding in the same way as the document embeddings.

- `query_embedding = query_embedding.astype("float32")` — Changes the question embedding into the numerical format required by FAISS.

- `search_k = min(top_k, index.ntotal)` — Prevents the program from requesting more results than the number of vectors stored in FAISS.

- `scores, indices = index.search(...)` — Searches the FAISS index using the question embedding.

- `query_embedding` — Provides the numerical version of the question.

- `search_k` — Specifies how many results should be returned.

- `scores` — Stores the similarity scores.

- `indices` — Stores the positions of the matching embeddings.

- `results = []` — Creates an empty list for the final retrieval results.

- `for rank, idx in enumerate(indices[0]):` — Goes through each returned FAISS result while keeping its ranking.

- `if idx == -1:` — Checks whether FAISS returned an invalid position.

- `continue` — Skips that result.

- `chunk = chunk_lookup[int(idx)]` — Uses the FAISS position to find the original chunk.

- `results.append({ ... })` — Adds the retrieved chunk information to the results.

- `"rank": rank + 1` — Stores the ranking starting from 1.

- `"chunk_id": chunk["chunk_id"]` — Stores the chunk ID.

- `"page": chunk["page"]` — Stores the original PDF page.

- `"score": float(scores[0][rank])` — Stores the similarity score.

- `"text": chunk["text"]` — Stores the actual chunk text.

- `return results` — Returns the completed list of retrieved chunks.

---

### **Why this step is important:**

This function performs the retrieval part of RAG. It finds the document sections that are most related to the user's question before the language model generates an answer.

---

In [150]:
# ============================================================
# 17. DEFINE RETRIEVAL FUNCTION
# ============================================================

def retrieve_documents(
    query,
    top_k=3
):
    """
    Retrieve the top matching chunks using FAISS.

    Returns chunk ID, page, similarity score, and text.
    """

    if not query or not query.strip():
        raise ValueError("The query cannot be empty.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # Convert the question into an embedding.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    # Avoid requesting more results than available vectors.
    search_k = min(top_k, index.ntotal)

    # Search the FAISS vector store.
    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        if idx == -1:
            continue

        chunk = chunk_lookup[int(idx)]

        results.append({
            "rank": rank + 1,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(scores[0][rank]),
            "text": chunk["text"]
        })

    return results

---
### **Cell 18: Test Retrieval**

**Purpose:**  
Allows the user to manually test whether the FAISS vector store can successfully retrieve relevant document chunks.

**Input:**  
- Test question entered by the user
- `TOP_K`
- Retrieval function from Cell 17

**Output:**  
- Top matching chunks
- Rank
- Chunk ID
- Page
- Similarity score
- Retrieved text

---

**Line-by-line breakdown:**

- `test_query = input(...)` — Asks the user to enter a test question.

- `"(or press Enter to skip)"` — Tells the user that the retrieval test is optional.

- `if test_query.strip():` — Continues only when the user actually entered a question.

- `test_results = retrieve_documents(` — Calls the retrieval function.

- `test_query` — Provides the user's question.

- `TOP_K` — Tells the function how many results to retrieve.

- `for result in test_results:` — Goes through every retrieved result.

- `print(f"\nRESULT {result['rank']}")` — Displays the result's ranking.

- `print(f"Chunk ID   : {result['chunk_id']}")` — Displays the chunk ID.

- `print(f"Page       : {result['page']}")` — Displays the original page.

- `print(f"Similarity : {result['score']:.4f}")` — Displays the similarity score rounded to four decimal places.

- `print(result["text"])` — Displays the actual retrieved text.

---

### **Why this step is important:**

Testing retrieval confirms that the vector store works before moving to the language-model portion of the RAG system.

---

In [151]:
# ============================================================
# 18. TEST RETRIEVAL
# ============================================================

print("\n")
print("=" * 70)
print("TESTING VECTOR STORE")
print("=" * 70)

test_query = input(
    "\nEnter a test question "
    "(or press Enter to skip): "
)

if test_query.strip():

    test_results = retrieve_documents(
        test_query,
        TOP_K
    )

    print("\n")
    print("=" * 70)
    print("RETRIEVED CHUNKS")
    print("=" * 70)

    for result in test_results:

        print(f"\nRESULT {result['rank']}")
        print(f"Chunk ID   : {result['chunk_id']}")
        print(f"Page       : {result['page']}")
        print(f"Similarity : {result['score']:.4f}")

        print("-" * 70)

        print(result["text"])



TESTING VECTOR STORE

Enter a test question (or press Enter to skip): 


---
### **Cell 19: Save FAISS Index**

**Purpose:**  
Saves the completed FAISS vector index into a file.

**Input:**  
- `index`

**Output:**  
- `rag_faiss.index`

---

**Line-by-line breakdown:**

- `FAISS_FILE = "rag_faiss.index"` — Defines the filename that will be used to save the FAISS index.

- `faiss.write_index(` — Calls FAISS's saving function.

- `index` — Provides the vector index that should be saved.

- `FAISS_FILE` — Provides the destination filename.

- `print("\nFAISS index saved:", FAISS_FILE)` — Confirms that the FAISS index was saved and displays its filename.

---

### **Why this step is important:**

Saving the FAISS index allows the created vector store to be kept instead of rebuilding it every time the program is restarted.

---

In [ ]:
# ============================================================
# 19. SAVE FAISS INDEX
# ============================================================

FAISS_FILE = "rag_faiss.index"

faiss.write_index(
    index,
    FAISS_FILE
)

print("\nFAISS index saved:", FAISS_FILE)


FAISS index saved: rag_faiss.index


---
### **Cell 20: Save Chunks and Metadata**

**Purpose:**  
Saves the generated document chunks into a Python file so their text, page numbers, and IDs can be loaded again later.

**Input:**  
- `chunks`

**Output:**  
- `rag_chunks.pkl`

---

**Line-by-line breakdown:**

- `CHUNKS_FILE = "rag_chunks.pkl"` — Defines the filename used to store the chunks.

- `with open(CHUNKS_FILE, "wb") as f:` — Opens the file in binary writing mode. `"wb"` means write binary.

- `pickle.dump(` — Uses Pickle to save Python data.

- `chunks` — Provides the list of chunks that should be saved.

- `f` — Specifies the opened file where the chunks should be written.

- `print("Chunks saved:", CHUNKS_FILE)` — Confirms that the chunk file was created.

---

### **Why this step is important:**

The FAISS index contains numerical embeddings, but the actual text and page information are stored separately. Saving the chunks makes it possible to reconnect search results with their original text.

---

In [ ]:
# ============================================================
# 20. SAVE CHUNKS AND METADATA
# ============================================================

CHUNKS_FILE = "rag_chunks.pkl"

with open(CHUNKS_FILE, "wb") as f:

    pickle.dump(
        chunks,
        f
    )

print("Chunks saved:", CHUNKS_FILE)

Chunks saved: rag_chunks.pkl


---
### **Cell 21: Save Indexing Configuration**

**Purpose:**  
Saves the settings used to create the current vector store.

**Input:**  
- Chunk size
- Chunk overlap
- Top-K
- Embedding model
- Source PDF filename

**Output:**  
- Configuration file named `rag_config.pkl`

---

**Line-by-line breakdown:**

- `CONFIG_FILE = "rag_config.pkl"` — Defines the filename for the saved configuration.

- `config = {` — Creates a collection containing the current settings.

- `"chunk_size": CHUNK_SIZE` — Saves the selected chunk size.

- `"chunk_overlap": CHUNK_OVERLAP` — Saves the selected overlap.

- `"top_k": TOP_K` — Saves the number of retrieval results.

- `"embedding_model": EMBEDDING_MODEL_NAME` — Saves the name of the embedding model.

- `"source_document": pdf_file` — Saves the original PDF filename.

- `with open(CONFIG_FILE, "wb") as f:` — Opens the configuration file in binary writing mode.

- `pickle.dump(config, f)` — Saves the configuration into the file.

- `print("Configuration saved:", CONFIG_FILE)` — Confirms that the configuration was saved.

---

### **Why this step is important:**

Saving the configuration makes it possible to identify exactly which settings were used to create a particular vector store. This is especially useful when comparing multiple chunking experiments.

---

In [ ]:
# ============================================================
# 21. SAVE INDEXING CONFIGURATION
# ============================================================

CONFIG_FILE = "rag_config.pkl"

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "source_document": pdf_file
}

with open(CONFIG_FILE, "wb") as f:

    pickle.dump(
        config,
        f
    )

print("Configuration saved:", CONFIG_FILE)

Configuration saved: rag_config.pkl


---
### **Cell 22: Indexing Complete**

**Purpose:**  
Displays a final summary showing that document processing and vector indexing were successfully completed.

**Input:**  
- Source PDF information
- Generated chunks
- FAISS index
- Chunking settings
- Saved filenames

**Output:**  
- Final indexing summary
- List of generated files
- Confirmation that the vector store is ready

---

**Line-by-line breakdown:**

- `print("INDEXING COMPLETE")` — Displays the completion message.

- `print("Source document :", pdf_file)` — Displays the PDF used.

- `print("Number of chunks:", len(chunks))` — Displays the total number of generated chunks.

- `print("Vectors stored  :", index.ntotal)` — Displays the total number of embeddings stored in FAISS.

- `print("Chunk size      :", CHUNK_SIZE)` — Displays the chunk-size setting.

- `print("Chunk overlap   :", CHUNK_OVERLAP)` — Displays the overlap setting.

- `print("Embedding model :", EMBEDDING_MODEL_NAME)` — Displays the embedding model used.

- `print("\nSaved files:")` — Introduces the list of generated files.

- `print("  ", EXCEL_FILE)` — Displays the generated Excel file.

- `print("  ", FAISS_FILE)` — Displays the saved FAISS index.

- `print("  ", CHUNKS_FILE)` — Displays the saved chunk information.

- `print("  ", CONFIG_FILE)` — Displays the saved configuration.

- `print("\nThe vector store is ready for retrieval.")` — Confirms that indexing was successfully completed.

- `print("Inspect the downloaded Excel workbook...")` — Reminds the user to manually inspect the generated chunks.

---

### **Why this step is important:**

This provides a clear ending for Component I and confirms the settings, number of chunks, stored vectors, and generated files before moving to question answering.

---

In [ ]:
# ============================================================
# 22. INDEXING COMPLETE
# ============================================================

print("\n")
print("=" * 70)
print("INDEXING COMPLETE")
print("=" * 70)

print("Source document :", pdf_file)
print("Number of chunks:", len(chunks))
print("Vectors stored  :", index.ntotal)
print("Chunk size      :", CHUNK_SIZE)
print("Chunk overlap   :", CHUNK_OVERLAP)
print("Embedding model :", EMBEDDING_MODEL_NAME)

print("\nSaved files:")
print("  ", EXCEL_FILE)
print("  ", FAISS_FILE)
print("  ", CHUNKS_FILE)
print("  ", CONFIG_FILE)

print("\nThe vector store is ready for retrieval.")
print(
    "Inspect the downloaded Excel workbook to evaluate "
    "the generated chunks."
)



INDEXING COMPLETE
Source document : AO-2026-0005-Guidelines-on-Differentiated-Service-Delivery-for-Pre-Exposure-Prophylaxis-PrEP-and-Post-Exposure-Prophylaxis-PEP-in-the-Prevention-of-HIV-in-the-Philippines-1 (1).pdf
Number of chunks: 80
Vectors stored  : 80
Chunk size      : 1000
Chunk overlap   : 100
Embedding model : all-MiniLM-L6-v2

Saved files:
   AO-2026-0005-Guidelines-on-Differentiated-Service-Delivery-for-Pre-Exposure-Prophylaxis-PrEP-and-Post-Exposure-Prophylaxis-PEP-in-the-Prevention-of-HIV-in-the-Philippines-1 (1)_chunks_inspection.xlsx
   rag_faiss.index
   rag_chunks.pkl
   rag_config.pkl

The vector store is ready for retrieval.
Inspect the downloaded Excel workbook to evaluate the generated chunks.


#**Component II: Prompt and Retrieval**

---

### **Cell 1: Import Required Libraries**

**Purpose:**  

Imports the tools needed to load Meta-Llama-3-8B-Instruct, authenticate with Hugging Face, use 4-bit quantization, process text, and run the language model using PyTorch.

**Input:**  

- Installed Hugging Face Transformers library

- Installed Hugging Face Hub library

- Installed PyTorch library

- Google Colab environment

**Output:**  

- The tokenizer and causal language model classes become available

- The 4-bit quantization configuration becomes available

- Google Colab Secrets can be accessed

- Hugging Face authentication becomes available

- PyTorch and regular expression tools are ready to use

---

**Line-by-line breakdown:**

- `from transformers import (` — Begins importing the required tools from the Hugging Face Transformers library.

- `AutoTokenizer,` — Imports the tokenizer loader. The tokenizer converts normal text into tokens that Meta-Llama-3-8B-Instruct can understand.

- `AutoModelForCausalLM,` — Imports the model class used for causal language models such as Llama 3. This replaces `AutoModelForSeq2SeqLM`, which was previously used for FLAN-T5.

- `BitsAndBytesConfig` — Imports the configuration tool used to set up 4-bit quantization. This helps reduce the amount of GPU memory needed by Llama 3.

- `)` — Ends the group of imports from the Transformers library.

- `from google.colab import userdata` — Imports the Google Colab tool used to securely retrieve private information stored in Colab Secrets. In this notebook, it is used to retrieve the Hugging Face access token.

- `from huggingface_hub import login` — Imports the Hugging Face login function. This allows the notebook to authenticate before accessing Meta-Llama-3-8B-Instruct.

- `import torch` — Imports PyTorch, which is used to run the language model and check whether a CUDA GPU is available.

- `import re` — Imports Python's regular expression library, which is used for text matching and processing in the RAG system.

---

### **Why this step is important:**

The previous FLAN-T5 model used `AutoModelForSeq2SeqLM`, while Meta-Llama-3-8B-Instruct uses `AutoModelForCausalLM`. The additional imports are also needed to authenticate with Hugging Face and load the larger Llama 3 model using 4-bit quantization.

Without these imports, the notebook would not be able to securely access, load, or run Meta-Llama-3-8B-Instruct.

---

In [ ]:
# ============================================================
# RAG PROMPT PROCESSING
# TOKEN-LIMITED CONTEXT, ANSWER GENERATION, AND DIAGNOSTICS
# Model: Meta-Llama-3-8B-Instruct
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from google.colab import userdata
from huggingface_hub import login

import torch
import re

---

### **Cell 2: Authenticate Hugging Face and Check Device**

**Purpose:**  

Connects the Google Colab notebook to the user's Hugging Face account and checks whether a CUDA-compatible GPU is available for running Meta-Llama-3-8B-Instruct.

**Input:**  

- Hugging Face access token stored in Google Colab Secrets as `HF_TOKEN`

- Google Colab runtime with GPU enabled

**Output:**  

- The notebook is authenticated with Hugging Face

- A CUDA GPU is confirmed to be available

- The selected device and GPU name are displayed

---

**Line-by-line breakdown:**

- `HF_TOKEN = userdata.get("HF_TOKEN")` — Retrieves the Hugging Face access token stored securely in Google Colab Secrets under the name `HF_TOKEN`. This prevents the actual token from being written directly inside the notebook.

- `if not HF_TOKEN:` — Checks whether the notebook was able to retrieve the Hugging Face token.

- `raise RuntimeError(...)` — Stops the program and displays an error message if `HF_TOKEN` cannot be found. This prevents the notebook from trying to access the gated Llama 3 model without authentication.

- `login(` — Starts the Hugging Face authentication process.

- `token=HF_TOKEN,` — Uses the token retrieved from Google Colab Secrets to authenticate the user's Hugging Face account.

- `add_to_git_credential=False` — Prevents the Hugging Face token from being added to the Git credential storage.

- `)` — Completes the Hugging Face login command.

- `if not torch.cuda.is_available():` — Checks whether Google Colab currently has access to a CUDA-compatible GPU.

- `raise RuntimeError(...)` — Stops the program if no CUDA GPU is available and tells the user to enable a GPU runtime.

- `device = "cuda"` — Sets CUDA as the device that will be used for language model processing.

- `print("=" * 70)` — Prints a separator line to make the notebook output easier to read.

- `print("RAG PROMPT PROCESSING")` — Displays the title of the RAG processing section.

- `print("Hugging Face authentication successful.")` — Displays a message confirming that Hugging Face authentication was completed.

- `print("Device:", device)` — Displays the selected processing device, which should be `cuda`.

- `print("GPU:", torch.cuda.get_device_name(0))` — Displays the name of the GPU assigned to the Google Colab runtime.

---

### **Why this step is important:**

Meta-Llama-3-8B-Instruct is stored in a gated Hugging Face repository, so the notebook must authenticate using an account that has permission to access the model.

The model is also much larger than FLAN-T5. Checking for a CUDA GPU before loading it prevents the notebook from attempting to run the 8-billion-parameter model using an unsupported setup.

---

In [ ]:
# ============================================================
# 2. AUTHENTICATE HUGGING FACE AND CHECK DEVICE
# ============================================================

# Store your Hugging Face access token in Google Colab Secrets
# using the name: HF_TOKEN
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found in Google Colab Secrets. "
        "Add your Hugging Face token as a secret named HF_TOKEN."
    )

login(token=HF_TOKEN, add_to_git_credential=False)

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required for this 4-bit Meta-Llama-3-8B-Instruct setup. "
        "In Google Colab, enable a GPU runtime before running Component II."
    )

device = "cuda"

print("=" * 70)
print("RAG PROMPT PROCESSING")
print("=" * 70)
print("Hugging Face authentication successful.")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

RAG PROMPT PROCESSING
Hugging Face authentication successful.
Device: cuda
GPU: Tesla T4


---

### **Cell 3: Load Meta-Llama-3-8B-Instruct**

**Purpose:**  

Loads the Meta-Llama-3-8B-Instruct tokenizer and language model that will generate answers from the evidence retrieved by the FAISS RAG system.

**Input:**  

- Hugging Face access token

- Model name `meta-llama/Meta-Llama-3-8B-Instruct`

- CUDA-compatible GPU

- Token limit settings

**Output:**  

- Meta-Llama-3-8B-Instruct tokenizer is loaded

- Meta-Llama-3-8B-Instruct model is loaded using 4-bit quantization

- The language model is ready to generate source-grounded answers

---

**Line-by-line breakdown:**

- `LLM_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"` — Stores the Hugging Face model name in the `LLM_NAME` variable. This replaces the previous `google/flan-t5-base` model.

- `MAX_INPUT_TOKENS = 512` — Sets the controlled maximum token budget used when building the RAG input. Keeping this value controlled helps maintain consistency during the lab experiment.

- `MAX_NEW_TOKENS = 128` — Sets the maximum number of new tokens that Llama 3 can generate for its answer. This was increased from 32 to 128 so that the model has enough space to provide complete answers.

- `RESERVED_TOKEN_MARGIN = 32` — Reserves additional token space when constructing the RAG prompt. This provides a safety margin before reaching the input limit.

- `print("\nLoading language model:")` — Displays a message showing that the language model loading process is starting.

- `print(LLM_NAME)` — Displays the exact name of the language model being loaded.

- `quantization_config = BitsAndBytesConfig(` — Creates the configuration that controls how the Llama 3 model will be loaded using 4-bit quantization.

- `load_in_4bit=True,` — Enables 4-bit loading. This greatly reduces the amount of GPU memory required to store the model.

- `bnb_4bit_quant_type="nf4",` — Uses the NF4 quantization format, which is designed for efficiently representing model weights using 4-bit values.

- `bnb_4bit_compute_dtype=torch.float16,` — Uses 16-bit floating-point values when performing calculations with the quantized model.

- `bnb_4bit_use_double_quant=True` — Enables an additional quantization step that can further reduce memory usage.

- `)` — Completes the 4-bit quantization configuration.

- `tokenizer = AutoTokenizer.from_pretrained(` — Begins loading the tokenizer associated with Meta-Llama-3-8B-Instruct.

- `LLM_NAME,` — Tells Hugging Face which model's tokenizer should be loaded.

- `token=HF_TOKEN` — Uses the authenticated Hugging Face token to access the gated model repository.

- `)` — Completes the tokenizer loading process.

- `if tokenizer.pad_token is None:` — Checks whether the Llama tokenizer has a separate padding token.

- `tokenizer.pad_token = tokenizer.eos_token` — Uses the end-of-sequence token as the padding token if a separate padding token is not available.

- `llm = AutoModelForCausalLM.from_pretrained(` — Begins loading Meta-Llama-3-8B-Instruct as a causal language model.

- `LLM_NAME,` — Specifies the model that should be downloaded and loaded.

- `token=HF_TOKEN,` — Uses the Hugging Face token to access the model files.

- `quantization_config=quantization_config,` — Applies the previously created 4-bit quantization settings.

- `device_map="auto",` — Allows Hugging Face to automatically place the model on the available GPU.

- `torch_dtype=torch.float16` — Uses 16-bit floating-point calculations where appropriate to help reduce memory usage.

- `)` — Completes the language model loading process.

- `llm.eval()` — Places the model in evaluation mode because it will be used for generating answers instead of being trained.

- `print("Language model loaded successfully.")` — Displays a confirmation message after the model has loaded.

- `print("Model:", LLM_NAME)` — Displays the name of the loaded model.

- `print("Loading mode: 4-bit quantization")` — Confirms that the model was loaded using the memory-saving 4-bit configuration.

---

### **Why this step is important:**

This cell replaces the original FLAN-T5 language model with the project's proposed Meta-Llama-3-8B-Instruct model.

Llama 3 contains approximately 8 billion parameters and requires much more memory than FLAN-T5. Using 4-bit quantization makes it more practical to run the model using the GPU resources available in Google Colab.

---

In [ ]:
# ============================================================
# 3. LOAD META-LLAMA-3-8B-INSTRUCT
# ============================================================

LLM_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

# Keep the original controlled input budget for the lab experiment.
MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 128

# Reserve space within the input budget for safety.
RESERVED_TOKEN_MARGIN = 32

print("\nLoading language model:")
print(LLM_NAME)

# Load the 8B model in 4-bit form to reduce Colab GPU memory usage.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME,
    token=HF_TOKEN
)

# Llama does not define a separate padding token by default.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    token=HF_TOKEN,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

llm.eval()

print("Language model loaded successfully.")
print("Model:", LLM_NAME)
print("Loading mode: 4-bit quantization")


Loading language model:
meta-llama/Meta-Llama-3-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Language model loaded successfully.
Model: meta-llama/Meta-Llama-3-8B-Instruct
Loading mode: 4-bit quantization


---

### **Cell 4: Define Token Counting Function**

**Purpose:**  

Creates a function that measures how many Llama 3 tokens are contained in a piece of text. This is used later to control how much retrieved evidence can be placed inside the RAG prompt.

**Input:**  

- A text string

- Meta-Llama-3-8B-Instruct tokenizer

**Output:**  

- The total number of tokens contained in the provided text

---

**Line-by-line breakdown:**

- `def count_tokens(text):` — Creates a reusable function named `count_tokens`. The function accepts a piece of text as its input.

- `return len(` — Returns the number of items produced by the tokenizer.

- `tokenizer.encode(` — Converts the provided text into numerical token IDs using the Meta-Llama-3-8B-Instruct tokenizer.

- `text,` — Provides the text that should be converted into tokens.

- `add_special_tokens=False,` — Prevents the tokenizer from automatically adding special model tokens during basic token counting. The required chat-related tokens will be added later when the final prompt is prepared.

- `truncation=False` — Prevents the tokenizer from shortening the text. This allows the function to measure its actual token count.

- `)` — Completes the tokenization process.

- `)` — Counts the number of token IDs and returns the result.

---

### **Why this step is important:**

Language models process tokens instead of normal words. The RAG system needs to measure the number of tokens used by the instructions, question, and retrieved evidence so that the final prompt remains within the controlled input budget.

Using the Llama 3 tokenizer also makes the token count match the language model that is actually being used in the experiment.

---

In [ ]:
# ============================================================
# 4. DEFINE TOKEN COUNTING FUNCTION
# ============================================================

def count_tokens(text):
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False,
            truncation=False
        )
    )

---
### **Cell 5: Prioritize Retrieved Documents**

**Purpose:**  
Sorts the retrieved document chunks so the chunks with the highest similarity scores are processed first.

**Input:**  
- User question
- Retrieved document chunks

**Output:**  
- Retrieved chunks sorted from highest to lowest similarity score

---

**Line-by-line breakdown:**

- `def prioritize_documents(question, retrieved_documents):` — Creates a function for ordering the retrieved chunks.

- `"""Preserve FAISS similarity ranking..."""` — Explains that chunks with higher FAISS similarity scores should appear first.

- `return sorted(` — Sorts the retrieved document list and returns the result.

- `retrieved_documents` — Provides the list that should be sorted.

- `key=lambda result: result.get("score", float("-inf"))` — Tells Python to sort each result according to its `score`. If a score is missing, it uses a very low value so that result is placed last.

- `reverse=True` — Sorts from the highest score to the lowest score instead of the normal lowest-to-highest order.

- `question` — The function accepts the question as an input, although the current sorting process does not directly use it.

---

### **Why this step is important:**

The prompt has a limited amount of space. Prioritizing the strongest matches helps ensure that the most relevant retrieved evidence is considered first.

---

In [ ]:
# ============================================================
# 5. PRIORITIZE RETRIEVED DOCUMENTS
# ============================================================

def prioritize_documents(question, retrieved_documents):
    """
    Preserve FAISS similarity ranking.

    Assumes higher scores mean greater similarity,
    as with normalized embeddings and IndexFlatIP.
    """

    return sorted(
        retrieved_documents,
        key=lambda result: result.get("score", float("-inf")),
        reverse=True
    )

---

### **Cell 6: Define the Token-Limited RAG Prompt**

**Purpose:**  

Creates the final source-grounded prompt by combining the user's question with the evidence retrieved from FAISS while controlling the number of tokens sent to Meta-Llama-3-8B-Instruct.

**Input:**  

- User question

- Retrieved document chunks from FAISS

- Token limit settings

- Meta-Llama-3-8B-Instruct tokenizer

**Output:**  

- A complete RAG prompt containing the instructions, question, and selected document evidence

- Retrieved evidence is kept within the available token budget

---

**Line-by-line breakdown:**

- `instruction = (` — Begins creating the main instruction that tells the language model how it should answer the question.

- `"Answer the question using only the supplied evidence from the document. "` — Tells Llama 3 that its answer must be based only on the evidence retrieved from the uploaded document.

- `"Do not use outside knowledge and do not invent information. "` — Tells the model not to use information that is not present in the retrieved evidence and not to create unsupported information.

- `"When comparing PrEP and PEP, keep their purpose, eligibility, timing, "` — Begins a specific instruction for questions that compare PrEP and PEP.

- `"dosing, duration, monitoring, and transition guidance separate and "` — Identifies the important information that should be clearly separated when comparing PrEP and PEP.

- `"state only what the evidence supports. "` — Tells the model to include only statements that can be supported by the retrieved document evidence.

- `"If the supplied evidence does not support an answer, respond exactly: "` — Tells the model what to do when the retrieved chunks do not contain enough information to answer the question.

- `"'I cannot find the answer in the provided document.'"` — Defines the exact fallback response that should be returned instead of guessing.

- `)` — Completes the source-grounded instruction.

- `prompt_suffix = "\n\nProvide a concise, source-grounded answer."` — Adds a final instruction asking Llama 3 to provide a short answer that is supported by the retrieved evidence.

- Token budget calculation — Calculates how many tokens are already being used by the instructions and question, then determines how much space remains for retrieved evidence.

- Retrieved evidence selection — Adds the most relevant document chunks while checking that they still fit within the available token budget.

- Final prompt construction — Combines the instructions, user's question, and selected document evidence into one final prompt that can be passed to the language model.

---

### **Why this step is important:**

This cell is important for the source-grounded design of the RAG system. It tells Meta-Llama-3-8B-Instruct to answer only from the evidence retrieved from the PrEP and PEP guideline instead of depending on its own outside knowledge.

The fallback response also helps reduce hallucination because the model is instructed to say that the answer cannot be found when the retrieved evidence is not enough.

---

In [ ]:
# ============================================================
# 6. DEFINE THE TOKEN-LIMITED RAG PROMPT
# ============================================================

def create_rag_prompt(question, retrieved_documents):
    """
    Build a source-grounded prompt that preserves the question
    and includes as much relevant retrieved evidence as the
    controlled input budget allows.
    """

    ranked_documents = prioritize_documents(
        question,
        retrieved_documents
    )

    instruction = (
        "Answer the question using only the supplied evidence from the document. "
        "Do not use outside knowledge and do not invent information. "
        "When comparing PrEP and PEP, keep their purpose, eligibility, timing, "
        "dosing, duration, monitoring, and transition guidance separate and "
        "state only what the evidence supports. "
        "If the supplied evidence does not support an answer, respond exactly: "
        "'I cannot find the answer in the provided document.'"
    )

    prompt_prefix = (
        f"Question: {question}\n\n"
        f"{instruction}\n\n"
        "Evidence:\n"
    )

    prompt_suffix = "\n\nProvide a concise, source-grounded answer."

    prefix_tokens = tokenizer.encode(
        prompt_prefix,
        add_special_tokens=False
    )

    suffix_tokens = tokenizer.encode(
        prompt_suffix,
        add_special_tokens=False
    )

    available_context_tokens = (
        MAX_INPUT_TOKENS
        - len(prefix_tokens)
        - len(suffix_tokens)
        - RESERVED_TOKEN_MARGIN
    )

    if available_context_tokens <= 0:
        raise ValueError(
            "The question and instructions leave no room for retrieved evidence. "
            "Shorten the question."
        )

    selected_parts = []
    remaining_tokens = available_context_tokens

    for result in ranked_documents:
        if remaining_tokens <= 0:
            break

        source_text = (
            f"[Page {result['page']}, Chunk {result['chunk_id']}]\n"
            f"{result['text']}"
        )

        source_tokens = tokenizer.encode(
            source_text,
            add_special_tokens=False
        )

        separator_tokens = (
            tokenizer.encode("\n\n", add_special_tokens=False)
            if selected_parts
            else []
        )

        tokens_available = remaining_tokens - len(separator_tokens)

        if tokens_available <= 0:
            break

        included_tokens = source_tokens[:tokens_available]

        if not included_tokens:
            continue

        included_text = tokenizer.decode(
            included_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        selected_parts.append(included_text)

        remaining_tokens -= (
            len(separator_tokens) + len(included_tokens)
        )

        if len(included_tokens) < len(source_tokens):
            break

    context = "\n\n".join(selected_parts)

    prompt = prompt_prefix + context + prompt_suffix

    final_token_count = count_tokens(prompt)

    if final_token_count > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Prompt contains {final_token_count} tokens, exceeding the "
            f"{MAX_INPUT_TOKENS}-token experiment limit."
        )

    print("\n--- PROMPT DIAGNOSTICS ---")
    print("Original question:", question)
    print("Retrieved chunks:", len(retrieved_documents))
    print("Chunks included:", len(selected_parts))
    print("Prompt content tokens:", final_token_count)
    print("Experiment input budget:", MAX_INPUT_TOKENS)

    print("\n--- SELECTED EVIDENCE ---")
    for i, part in enumerate(selected_parts, start=1):
        print(f"\nEvidence {i}:")
        print(part)

    return prompt

---

### **Cell 7: Define Answer Generation Function**

**Purpose:**  

Creates the function that sends the source-grounded RAG prompt to Meta-Llama-3-8B-Instruct and generates the final answer.

**Input:**  

- Completed RAG prompt

- Meta-Llama-3-8B-Instruct tokenizer

- Loaded Meta-Llama-3-8B-Instruct model

- Maximum number of new output tokens

**Output:**  

- A generated answer based on the retrieved document evidence

---

**Line-by-line breakdown:**

- `def generate_answer(prompt, max_new_tokens=MAX_NEW_TOKENS):` — Creates a reusable function named `generate_answer`. It accepts the completed RAG prompt and the maximum number of tokens that the model is allowed to generate.

- `messages = [` — Begins creating the chat message structure expected by Meta-Llama-3-8B-Instruct.

- `{"role": "system", ...}` — Creates the system message. A system message gives the language model its main behavior and rules before it receives the user's request.

- `"You are a source-grounded RAG assistant. "` — Defines the model's role as an assistant that should answer using retrieved evidence.

- `"Use only the evidence supplied by the user. "` — Tells the model that its answer should only use the evidence included in the RAG prompt.

- `"Do not guess and do not use outside knowledge. "` — Prevents the model from intentionally adding information that is not supported by the retrieved evidence.

- `"If the evidence is insufficient, say exactly: "` — Tells the model what it should do when the provided evidence cannot answer the question.

- `"'I cannot find the answer in the provided document.'"` — Defines the exact fallback response for insufficient evidence.

- `{"role": "user", "content": prompt}` — Places the completed RAG prompt into the user message. This prompt already contains the question and retrieved evidence.

- `]` — Completes the chat message structure.

- `inputs = tokenizer.apply_chat_template(` — Uses Llama 3's chat template to format the system and user messages in the structure expected by the instruction-tuned model.

- `messages,` — Provides the previously created system and user messages.

- `add_generation_prompt=True,` — Adds the required marker that tells Llama 3 that it should begin generating the assistant's response.

- `tokenize=True,` — Converts the formatted chat messages into numerical tokens.

- `return_dict=True,` — Returns the tokenized information in a dictionary containing items such as the input token IDs.

- `return_tensors="pt"` — Converts the tokenized data into PyTorch tensors so it can be processed by the language model.

- `)` — Completes the chat template and tokenization process.

- `input_length = inputs["input_ids"].shape[1]` — Measures how many tokens are contained in the complete chat-formatted input.

- `print("\n--- GENERATION DIAGNOSTICS ---")` — Displays a heading for information about the generation process.

- `print("Actual chat-formatted input tokens:", input_length)` — Displays the actual number of input tokens after Llama 3's chat formatting has been applied.

- `model_limit = tokenizer.model_max_length` — Retrieves the maximum token limit reported by the tokenizer.

- `if (...)` — Checks whether the final chat-formatted input is larger than the supported model limit.

- `raise ValueError(...)` — Stops generation and displays an error if the input exceeds the supported limit.

- `model_device = next(llm.parameters()).device` — Finds the device where the Llama 3 model is currently loaded.

- `inputs = { ... }` — Prepares a new version of the input dictionary.

- `value.to(model_device)` — Moves each input tensor to the same device as the language model.

- `with torch.no_grad():` — Disables gradient calculations because the model is only being used to generate answers and is not being trained.

- `outputs = llm.generate(` — Starts the answer generation process using Meta-Llama-3-8B-Instruct.

- `**inputs,` — Sends the tokenized chat prompt into the language model.

- `max_new_tokens=max_new_tokens,` — Limits the maximum number of new tokens that the model can generate.

- `do_sample=False,` — Disables random sampling. This makes the model's generation more consistent, which is useful when conducting a controlled experiment.

- `pad_token_id=tokenizer.eos_token_id,` — Uses the end-of-sequence token as the padding token during generation.

- `eos_token_id=tokenizer.eos_token_id` — Tells the model that generation can stop when the end-of-sequence token is produced.

- `)` — Completes the model generation command.

- `generated_tokens = outputs[0][input_length:]` — Removes the original input tokens from the output and keeps only the new tokens generated by Llama 3.

- `answer = tokenizer.decode(` — Begins converting the generated token IDs back into normal readable text.

- `generated_tokens,` — Provides the newly generated answer tokens.

- `skip_special_tokens=True` — Removes special model tokens that should not appear in the final answer.

- `).strip()` — Removes unnecessary spaces or blank lines from the beginning and end of the generated answer.

- `if not answer:` — Checks whether the language model returned an empty answer.

- `return (...)` — Returns an error message if no answer was generated.

- `return answer` — Returns the final generated answer to the rest of the RAG system.

---

### **Why this step is important:**

Meta-Llama-3-8B-Instruct works differently from the previous FLAN-T5 model. Llama 3 Instruct expects its instructions and questions to follow a chat format, so `apply_chat_template()` is used before generation.

Using `do_sample=False` also makes the answers more consistent between experiments. This is useful for the lab because changes in the results should mainly come from the tested RAG configuration rather than random differences in language model generation.

---

In [ ]:
# ============================================================
# 7. DEFINE ANSWER GENERATION FUNCTION
# ============================================================

def generate_answer(prompt, max_new_tokens=MAX_NEW_TOKENS):
    """
    Generate a deterministic answer using Llama 3's chat template.
    Only newly generated tokens are decoded as the final answer.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "You are a source-grounded RAG assistant. "
                "Use only the evidence supplied by the user. "
                "Do not guess and do not use outside knowledge. "
                "If the evidence is insufficient, say exactly: "
                "'I cannot find the answer in the provided document.'"
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    input_length = inputs["input_ids"].shape[1]

    print("\n--- GENERATION DIAGNOSTICS ---")
    print("Actual chat-formatted input tokens:", input_length)

    # The chat template adds a small number of special/system tokens.
    # The evidence itself is still controlled by MAX_INPUT_TOKENS above.
    model_limit = tokenizer.model_max_length
    if model_limit and model_limit < 100000 and input_length > model_limit:
        raise ValueError(
            f"Chat-formatted input contains {input_length} tokens, "
            f"but the tokenizer limit is {model_limit}."
        )

    model_device = next(llm.parameters()).device
    inputs = {
        key: value.to(model_device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        return (
            "The model generated an empty answer. "
            "Inspect the prompt and retrieved evidence."
        )

    return answer

---
### **Cell 8: Define Complete RAG Query Function**

**Purpose:**  
Combines retrieval, prompt creation, and answer generation into one complete question-answering function.

**Input:**  
- User question
- `top_k` number of retrieval results

**Output:**  
- Generated answer
- Retrieved document chunks
- Final prompt sent to the language model

---

**Line-by-line breakdown:**

- `def rag_query(question, top_k=3):` — Creates the main RAG question-answering function.

- `""" ... """` — Describes the complete workflow performed by the function.

- `question = question.strip()` — Removes unnecessary spaces from the beginning and end of the question.

- `if not question:` — Checks whether anything remains after removing spaces.

- `return ("Please enter a valid question.", [], "")` — Returns a message and empty results when the question is invalid.

- `retrieved_documents = retrieve_documents(` — Calls the retrieval function created in Component I.

- `question` — Provides the user's question.

- `top_k=top_k` — Provides the requested number of retrieval results.

- `if not retrieved_documents:` — Checks whether retrieval returned any evidence.

- `return ("I cannot find the answer in the provided document.", [], "")` — Returns the fallback message when no evidence is available.

- `prompt = create_rag_prompt(` — Calls the prompt-building function.

- `question` — Provides the original question.

- `retrieved_documents` — Provides the matching document chunks.

- `answer = generate_answer(prompt)` — Sends the completed prompt to the language model and generates an answer.

- `if not answer or not answer.strip():` — Checks whether the model returned an empty answer.

- `answer = (...)` — Replaces an empty answer with an explanatory message.

- `return (` — Begins returning all final results.

- `answer.strip()` — Returns the cleaned answer.

- `retrieved_documents` — Returns the document evidence used during retrieval.

- `prompt` — Returns the complete prompt that was sent to FLAN-T5.

---

### **Why this step is important:**

This function connects the major parts of the RAG pipeline. Instead of manually calling retrieval, prompt creation, and answer generation separately, one function performs the entire process.

---

In [ ]:
# ============================================================
# 8. DEFINE COMPLETE RAG QUERY FUNCTION
# ============================================================


def rag_query(question, top_k=3):
    """
    General-purpose RAG question-answering function.

    Workflow:
    1. Retrieve relevant document chunks.
    2. Build a token-limited prompt.
    3. Generate an answer using the language model.
    4. Return the answer, retrieved sources, and prompt.
    """

    # STEP 1: Validate the question.
    question = question.strip()

    if not question:
        return ("Please enter a valid question.", [], "")


    # STEP 2: Retrieve relevant document chunks.
    retrieved_documents = retrieve_documents(
        question,
        top_k=top_k
    )

    # STEP 3: Handle cases where no evidence is retrieved.
    if not retrieved_documents:
        return (
            "I cannot find the answer in the provided document.",
            [],
            ""
        )

    # STEP 4: Construct the token-limited RAG prompt.
    prompt = create_rag_prompt(
        question,
        retrieved_documents
    )

    # STEP 5: Generate an answer from the retrieved evidence.
    answer = generate_answer(prompt)

    # STEP 6: Handle an empty model response.
    if not answer or not answer.strip():
        answer = (
            "The model could not generate an answer. "
            "Please review the retrieved evidence."
        )

    # STEP 7: Return the results.
    return (
        answer.strip(),
        retrieved_documents,
        prompt
    )

---
### **Cell 9: Continuous Question-Answer Loop**

**Purpose:**  
Creates an interactive question-answering system that repeatedly accepts questions until the user chooses to exit. It also allows the user to inspect the sources and prompt from the previous question.

**Input:**  
- Questions entered by the user
- Special commands: `exit`, `sources`, and `prompt`

**Output:**  
- Generated RAG answers
- Retrieved source chunks
- Prompt sent to the language model
- Continuous interaction until `exit` is entered

---

**Line-by-line breakdown:**

- `print("CONTINUOUS RAG QUESTION ANSWERING")` — Displays the title of the interactive section.

- `print("Ask questions about the indexed document.")` — Tells the user what type of questions can be entered.

- `print("Type 'exit' to terminate.")` — Explains how to stop the program.

- `print("Type 'sources' to display sources from the last query.")` — Explains the command for viewing previous retrieved evidence.

- `print("Type 'prompt' to display the last generated prompt.")` — Explains the command for viewing the previous RAG prompt.

- `last_answer = None` — Creates a variable for storing the most recent answer.

- `last_sources = []` — Creates an empty list for the most recent retrieved sources.

- `last_prompt = None` — Creates a variable for storing the most recent prompt.

- `while True:` — Starts a loop that continues until it is manually stopped with `break`.

- `user_question = input("Question: ").strip()` — Asks the user for a question and removes unnecessary spaces.

- `if user_question.lower() == "exit":` — Checks whether the user entered `exit`. `.lower()` allows forms such as `EXIT` or `Exit` to work.

- `print("\nRAG session terminated.")` — Displays a termination message.

- `break` — Ends the continuous loop.

- `if not user_question:` — Checks whether the user entered an empty question.

- `print("Please enter a question.")` — Displays a warning.

- `continue` — Returns to the beginning of the loop without processing the empty question.

- `if user_question.lower() == "sources":` — Checks whether the user wants to see the sources from the previous query.

- `if not last_sources:` — Checks whether previous sources are available.

- `print("No previous query sources are available.")` — Explains that there are no stored sources yet.

- `continue` — Returns to the beginning of the loop.

- `print("SOURCES FROM LAST QUERY")` — Displays the source section title.

- `for result in last_sources:` — Goes through each source from the previous query.

- `f"Rank {result['rank']}"` — Displays the source's retrieval ranking.

- `f"Page {result['page']}"` — Displays the PDF page.

- `f"Chunk {result['chunk_id']}"` — Displays the chunk ID.

- `f"Similarity {result['score']:.4f}"` — Displays the similarity score rounded to four decimal places.

- `print(result["text"])` — Displays the actual retrieved source text.

- `if user_question.lower() == "prompt":` — Checks whether the user wants to see the previous prompt.

- `if last_prompt is None:` — Checks whether a previous prompt exists.

- `print("No previous query prompt is available.")` — Displays a message when there is no previous prompt.

- `print("LAST RAG PROMPT")` — Displays the prompt section title.

- `print(last_prompt)` — Displays the complete previous prompt.

- `try:` — Begins a protected section so errors can be handled without immediately crashing the interactive program.

- `answer, retrieved_documents, prompt = rag_query(` — Sends the user's question through the complete RAG pipeline.

- `user_question` — Provides the question.

- `top_k=TOP_K` — Uses the retrieval setting defined earlier.

- `last_answer = answer` — Saves the newest answer.

- `last_sources = retrieved_documents` — Saves the newest retrieved sources.

- `last_prompt = prompt` — Saves the newest prompt.

- `print("RAG ANSWER")` — Displays the answer section heading.

- `print(answer)` — Displays the language model's answer.

- `print("RETRIEVED SOURCES")` — Displays the source section heading.

- `for result in retrieved_documents:` — Goes through every retrieved source for the current question.

- `print(...)` — Displays the rank, page, chunk ID, and similarity score.

- `print(result["text"])` — Displays the retrieved chunk text.

- `print("RAG PROMPT SENT TO LLM")` — Displays the prompt section heading.

- `print(prompt)` — Displays the complete prompt that was sent to the language model.

- `except Exception as e:` — Catches an error that happens while processing the question and stores its information in `e`.

- `print("\nERROR:")` — Displays an error heading.

- `print(str(e))` — Displays the actual error message.

- `print("\nThe question could not be processed.")` — Tells the user that the current question failed.

---

### **Why this step is important:**

This turns the separate RAG functions into an interactive program. The user can ask multiple questions without rerunning the code and can inspect the retrieved evidence and generated prompt for each query.

The `sources` and `prompt` commands are also useful when evaluating whether the system retrieved the correct information and whether the language model received appropriate evidence.

---

In [152]:
# ============================================================
# 9. CONTINUOUS QUESTION-ANSWER LOOP
# ============================================================

print("\n")
print("=" * 70)
print("CONTINUOUS RAG QUESTION ANSWERING")
print("=" * 70)

print("Ask questions about the indexed document.")
print("Type 'exit' to terminate.")
print("Type 'sources' to display sources from the last query.")
print("Type 'prompt' to display the last generated prompt.")


# Store the most recent query results.
last_answer = None
last_sources = []
last_prompt = None


while True:

    print("\n")

    user_question = input(
        "Question: "
    ).strip()

    # --------------------------------------------------------
    # EXIT CONDITION
    # --------------------------------------------------------

    if user_question.lower() == "exit":

        print("\nRAG session terminated.")
        break

    # --------------------------------------------------------
    # IGNORE EMPTY QUESTIONS
    # --------------------------------------------------------

    if not user_question:

        print("Please enter a question.")
        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: SOURCES
    # --------------------------------------------------------

    if user_question.lower() == "sources":

        if not last_sources:
            print("No previous query sources are available.")
            continue

        print("\n" + "=" * 70)
        print("SOURCES FROM LAST QUERY")
        print("=" * 70)

        for result in last_sources:

            print(
                f"\nRank {result['rank']} | "
                f"Page {result['page']} | "
                f"Chunk {result['chunk_id']} | "
                f"Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: PROMPT
    # --------------------------------------------------------

    if user_question.lower() == "prompt":

        if last_prompt is None:
            print("No previous query prompt is available.")
            continue

        print("\n" + "=" * 70)
        print("LAST RAG PROMPT")
        print("=" * 70)

        print(last_prompt)

        continue

    # --------------------------------------------------------
    # PROCESS QUESTION
    # --------------------------------------------------------

    try:

        answer, retrieved_documents, prompt = rag_query(
            user_question,
            top_k=TOP_K
        )

        last_answer = answer
        last_sources = retrieved_documents
        last_prompt = prompt

        # ----------------------------------------------------
        # DISPLAY ANSWER
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG ANSWER")
        print("=" * 70)

        print(answer)

        # ----------------------------------------------------
        # DISPLAY RETRIEVED SOURCES
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RETRIEVED SOURCES")
        print("=" * 70)

        for result in retrieved_documents:

            print(
                f"\n[{result['rank']}] "
                f"Page {result['page']} "
                f"| Chunk {result['chunk_id']} "
                f"| Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        # ----------------------------------------------------
        # DISPLAY PROMPT
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG PROMPT SENT TO LLM")
        print("=" * 70)

        print(prompt)

    except Exception as e:

        print("\nERROR:")
        print(str(e))

        print("\nThe question could not be processed.")



CONTINUOUS RAG QUESTION ANSWERING
Ask questions about the indexed document.
Type 'exit' to terminate.
Type 'sources' to display sources from the last query.
Type 'prompt' to display the last generated prompt.


Question: Within how many hours after a possible HIV exposure should PEP be taken, and how long should it be continued?


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- PROMPT DIAGNOSTICS ---
Original question: Within how many hours after a possible HIV exposure should PEP be taken, and how long should it be continued?
Retrieved chunks: 3
Chunks included: 3
Prompt content tokens: 479
Experiment input budget: 512

--- SELECTED EVIDENCE ---

Evidence 1:
[Page 26, Chunk 110]
exposure: b. If 4-5 PrEP doses were taken in the 7 days prior to exposure, no need for PEP. Instead, continue daily PrEP. c. If0-3 PrEP doses were taken in the 7 days prior to exposure, consider PEP. B. For all other groups and any type of exposure to HIV: 1. If 6-7 out of 7 PrEP doses were taken in the 7 days prior to exposure, no need for PEP, instead continue daily PrEP. 2. If0-5 out of 5 PrEP doses were taken in the 7 days prior to exposure, consider PEP.

Evidence 2:
[Page 9, Chunk 47]
s. Dolutegravir (DTG) is recommended as the preferred third drug, dosed accordingly When available, atazanavir/ritonavir (ATV/r), darunavir/ritonavir (DRV/r), lopinavir/ritonavir (LPV/r) and 

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- PROMPT DIAGNOSTICS ---
Original question: If a person has a known or suspected HIV exposure within the past 72 hours, should they be given PrEP or PEP, and why?
Retrieved chunks: 3
Chunks included: 3
Prompt content tokens: 479
Experiment input budget: 512

--- SELECTED EVIDENCE ---

Evidence 1:
[Page 6, Chunk 29]
iv. Persons with recent PEP use for sexual exposure to HIV, especially for persons who have used PEP more than once; v. People who inject drugs (PWID) and people who use drugs (PWUD); vi. Persons who request PrEP shall be given priority as itmay indicate the likelihood of HIV exposure; and vii. | Persons who screen non-reactive and test negative for HIV shall be routinely offered PrEP. b. PrEP is NOT appropriate and shall not be given to: i. People living with HIV; ii. People with a known or suspected HIV exposure within the past 72 hours (e.g., due to sexual assault, occupational exposure, or

Evidence 2:
[Page 26, Chunk 110]
exposure: b. If 4-5 PrEP doses were taken in 

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- PROMPT DIAGNOSTICS ---
Original question: After completing a 28-day PEP regimen, under what conditions can a person with ongoing HIV exposure risk transition directly to PrEP without a gap?
Retrieved chunks: 3
Chunks included: 3
Prompt content tokens: 479
Experiment input budget: 512

--- SELECTED EVIDENCE ---

Evidence 1:
[Page 10, Chunk 50]
C. Transitioning from PEP to PrEP and from PrEP to PEP a. Immediate transition to PrEP is preferable for persons with ongoing exposure to HIV. b. People who complete the 28-day PEP regimen and wish to use PrEP can start PrEP without a gap if they have a negative HIV test result upon completion of PEP and do not have any contraindications to the chosen PrEP product. c. Full transition protocol from PrEP to PEP is elaborated in Annex J. D. Differentiated Service Delivery for PrEP and PEP a. Sites of delivery shall be diversified to include health facilities, fixed and mobile community settings,

Evidence 2:
[Page 16, Chunk 77]
Persons who are r

---
### **Cell 10: End Program**

**Purpose:**  
Displays a final message after the continuous RAG question-answering loop has ended.

**Input:**  
- The question-answer loop must first be ended using the `exit` command

**Output:**  
- Final program completion message

---

**Line-by-line breakdown:**

- `print("\n")` — Adds an empty line before the final message.

- `print("=" * 70)` — Displays a separator consisting of 70 equal signs.

- `print("RAG PROGRAM FINISHED")` — Displays the final message showing that the RAG program has ended.

- `print("=" * 70)` — Displays another separator to clearly mark the end of the program.

---

### **Why this step is important:**

This provides a clear ending to the notebook's interactive RAG process. It confirms that the question-answering loop was intentionally terminated and the program has finished running.

---

In [137]:
# ============================================================
# 10. END
# ============================================================

print("\n")
print("=" * 70)
print("RAG PROGRAM FINISHED")
print("=" * 70)




RAG PROGRAM FINISHED
